# Dump dictionary references in Python mode and in JSON mode

A Python-mode dump keeps each relationship wrapper with its canonical members; a JSON-mode dump turns each selected member into its reference.

Use the repository environment for this branch. Run cells in order in a fresh kernel; no published-package install is needed.

## Import the types

In [ ]:
!pip install -q aoa-action-machine

In [1]:
from typing import Annotated

from pydantic import Field

from aoa.action_machine.domain import AggregateMany, BaseCoreEntity, BaseEnumEntity, CompositeOne, NoInverse, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources.base_storage import BaseStorage
from aoa.action_machine.resources import InMemoryDataSource

## Declare the model

In [2]:
class PricingDomain(BaseDomain):
    """Group the pricing declarations."""

    name = "pricing"
    description = "Pricing"


@meta(description="Where PricingStorage keeps its rows")
class PricingDataSource(InMemoryDataSource):
    """The store PricingStorage works with."""


@meta(description="Stores the pricing entities", domain=PricingDomain)
class PricingStorage(BaseStorage[PricingDataSource]):
    """Hold the pricing entities."""


@entity(description="Currencies", resource=PricingStorage)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare canonical currency values."""

    symbol: str = Field(description="Currency symbol")
    EUR = value(code="eur", display_name="Euro", symbol="€")
    USD = value(code="usd", display_name="US dollar", symbol="$")


@entity(description="Prices", resource=PricingStorage)
class PricesCoreEntity(BaseCoreEntity):
    """Select one primary currency and several alternatives."""

    primary: Annotated[CompositeOne[CurrencyEnumEntity], NoInverse()] = Field(description="Primary currency")
    alternatives: Annotated[AggregateMany[CurrencyEnumEntity], NoInverse()] = Field(description="Alternative currencies")


## Dump in Python mode

In [3]:
price = PricesCoreEntity(primary=CompositeOne(id="eur"), alternatives=AggregateMany(ids=("usd",)))

python_dump = price.model_dump()
print("Python primary:", python_dump["primary"])
print("Python alternatives:", python_dump["alternatives"])
print("Canonical member kept:", python_dump["primary"].entity is CurrencyEnumEntity.EUR)


Python primary: CompositeOne(id='eur', loaded)
Python alternatives: AggregateMany(count=1, entities_loaded=yes, n_entities=1)
Canonical member kept: True


## Dump in JSON mode

In [4]:
json_dump = price.model_dump(mode="json")
print("JSON primary:", json_dump["primary"])
print("JSON alternatives:", json_dump["alternatives"])


JSON primary: {'code': 'eur', 'display_name': 'Euro', 'properties': {'symbol': '€'}}
JSON alternatives: [{'code': 'usd', 'display_name': 'US dollar', 'properties': {'symbol': '$'}}]


Python mode is for code that keeps working with the entity: the wrappers and the canonical members stay as they are. JSON mode is for the wire: each member becomes its `to_reference()` form.